# 01 — Consistent SAM1 crown predictions for all Baego ROIs

**Branch:** `kvanoost/Musanga:v2` · **Folder:** `Crown/`  
**Output:** `MyDrive/Colab Notebooks/Musanga/V2/Crown/sam1_core_centroid_halo40_v2/`

## Scientific objective

Create one **consistent six-ROI SAM1 ViT-H dataset** for downstream ecological metrics (crown density, sizes, summed crown projected area and union canopy cover).

**Revised selection rule:** SAM polygons are retained when their centroids lie inside their 40 m processing core. **They are not filtered by ROI centroid.** A one-core (40 m) perimeter halo is processed around each ROI to capture crowns that overlap the ROI while being centred outside it. Per-ROI crown-count selection and polygon clipping occur later in Notebook 00.

**Provenance:** This is a new, versioned reproduction of the recovered Block 53 approach, not a claim of exact reproduction. The official Meta SAM1 ViT-H implementation is used; the ROI 1 pilot must be reviewed before producing all six ROIs. No training.

**Safety:** All outputs go to a fresh folder. The earlier `sam1_block53_reproduction/` pilot and historical Block 53 data remain untouched. Inference saves progress per core; changes to inference settings require a *new output version*.

**Runtime:** GPU for inference; Section 06B geometry audit is CPU-only. Sections are numbered and include clear outputs.

## 01 — Runtime, inputs, frozen configuration and run gate

Start with a **GPU runtime**. Re-run the ROI 1 pilot in the new versioned output directory, because previously saved tiles were filtered using the old ROI-centroid rule.

To prevent mixing incompatible results, processing all six ROIs requires explicit approval after inspecting the pilot. ROI 1 is part of the final six-ROI set; ROI 4 will also be generated using this same new method.

In [ ]:
# 01 — Setup
!pip -q install "segment-anything>=1.0" geopandas rasterio shapely

from pathlib import Path
import time, json, math, os, warnings
import numpy as np, pandas as pd, geopandas as gpd
import rasterio
from rasterio.windows import from_bounds, Window
from rasterio.features import shapes
from shapely.geometry import shape, box, Polygon, MultiPolygon
from shapely.ops import unary_union
from shapely import make_valid
from google.colab import drive
import torch

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
DATA = ROOT / "data/Geo_Ref"
SOURCE = ROOT / "KVO_work/KVO_06_UNet_guided_SAM3/experiments_v53_luki_sam_only_replication"
OUTPUT = ROOT / "V2/Crown/sam1_core_centroid_halo40_v2"
RGB_PATH = DATA / "Baego_georef.tif"
ROI_PATH = DATA / "Annotations/ROI_Crowns_Kvo.shp"
REFERENCE = DATA / "Annotations/Crown_Kristof.shp"

PILOT_ROI = 1
ALL_ROIS = tuple(range(6))
# LEAVE FALSE until section 06 has been evaluated.
RUN_ALL_ROIS = False
# For full benchmark reruns set this to True only after reviewing pilot discrepancy.
PILOT_APPROVED = False
# Do not silently overwrite previous outputs if parameters change.
CHECKPOINT_PATH = Path("/content/sam_vit_h_4b8939.pth")

CONFIG = dict(model="sam_vit_h", points_per_side=32,
              pred_iou_thresh=0.88, stability_score_thresh=0.95,
              core_m=40.0, context_m=60.0,
              min_pred_area_m2=2.0, max_pred_area_m2=2500.0,
              max_components=3, polygon_nms_iou=0.60,\n              roi_halo_m=40.0, selection="centroid_inside_core_only",\n              pipeline_version="sam1_core_centroid_halo40_v2")
if not torch.cuda.is_available():
    raise RuntimeError("GPU required for SAM1 ViT-H. Select Runtime > Change runtime type > T4 GPU.")
for path in (RGB_PATH, ROI_PATH, REFERENCE, SOURCE):
    if not path.exists(): raise FileNotFoundError(path)
OUTPUT.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUTPUT)
print("Frozen settings:", CONFIG)

## 02 — Input geometry, 40 m processing cores and exterior halo

The grid retains the original 40 m core spacing and coordinate origin, but also processes a **40 m halo around each ROI**. This allows crowns centred just outside the ROI to contribute their intersecting footprint to canopy cover.

Crown inclusion at inference is **core-only**. Notebook 00 subsequently counts crowns by ROI centroid, and computes summed CPA and union canopy cover from **all polygons intersecting the ROI**, clipped spatially to its footprint.

In [ ]:
# 02 — Validated ROI preparation
def ensure_crs(g, raster_crs, raster_bounds):
    if g.crs is not None:
        return g.to_crs(raster_crs)
    b = g.total_bounds; r = tuple(raster_bounds)
    if not (b[2]>=r[0] and b[0]<=r[2] and b[3]>=r[1] and b[1]<=r[3]):
        raise ValueError("Missing CRS with coordinates not overlapping the raster")
    print("WARNING: missing ROI CRS; raw coordinates overlap raster. Assigning raster CRS.")
    return g.set_crs(raster_crs, allow_override=True)

with rasterio.open(RGB_PATH) as raster:
    CRS, BOUNDS, RES, COUNT, DTYPE = raster.crs, raster.bounds, raster.res, raster.count, raster.dtypes[0]
if CRS is None or not CRS.is_projected:
    raise ValueError("Projected metre-based CRS required")
rois = ensure_crs(gpd.read_file(ROI_PATH), CRS, BOUNDS)
print("Raster:", RGB_PATH.name, "CRS:", CRS, "resolution:", RES,
      "bands:", COUNT, "dtype:", DTYPE)
print("ROI count:",len(rois), "requested:", ALL_ROIS)
assert len(rois)>=6
assert COUNT >= 3

def cleaned_polygon(g):
    if g is None or g.is_empty: return Polygon()
    if not g.is_valid: g = make_valid(g)
    if g.geom_type in ("Polygon","MultiPolygon"): return g
    parts = [p for p in getattr(g,"geoms",[])
             if p.geom_type in ("Polygon","MultiPolygon") and not p.is_empty]
    return unary_union(parts) if parts else Polygon()

def grid_for_roi(roi_id):
    region = cleaned_polygon(rois.geometry.iloc[roi_id])
    xmin,ymin,xmax,ymax = region.bounds
    step = CONFIG["core_m"]
    halo = CONFIG["roi_halo_m"]
    # Keep the original grid's alignment: core boundaries begin at xmin/ymin.
    # Include one complete ring of 40 m cores outside the ROI.
    nx = math.ceil((xmax - xmin) / step)
    ny = math.ceil((ymax - ymin) / step)
    ring = math.ceil(halo / step)
    target = region.buffer(halo)
    cores = []
    for iy in range(-ring, ny + ring):
        for ix in range(-ring, nx + ring):
            core = box(xmin + ix*step, ymin + iy*step,
                       xmin + (ix+1)*step, ymin + (iy+1)*step)
            if core.intersection(target).area > 0:
                cores.append(core)
    return cores

for i in ALL_ROIS:
    print(f"ROI {i}: area={rois.geometry.iloc[i].area/10000:.3f} ha, "
          f"core windows={len(grid_for_roi(i))}")

## 03 — Initialize frozen SAM1 ViT-H

Load the official SAM1 ViT-H checkpoint once. The model is used in **automatic mask generation** mode; there is no fitting, retraining or GPU-intensive parameter grid.

This notebook uses the official `segment-anything` mask generator and **explicitly records processing decisions that may differ from the original Block 53 code**.

In [ ]:
# 03 — Checkpoint and SAM1 automatic mask generator
if not CHECKPOINT_PATH.is_file():
    import urllib.request
    url="https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    print("Downloading official SAM1 ViT-H checkpoint (~2.6 GB) to Colab local disk...")
    urllib.request.urlretrieve(url, CHECKPOINT_PATH)
    print("Checkpoint ready:", CHECKPOINT_PATH.stat().st_size, "bytes")
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

sam=sam_model_registry["vit_h"](checkpoint=str(CHECKPOINT_PATH))
sam.to(device="cuda")
sam.eval()
generator=SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=CONFIG["points_per_side"],
    pred_iou_thresh=CONFIG["pred_iou_thresh"],
    stability_score_thresh=CONFIG["stability_score_thresh"],
    crop_n_layers=0,
    output_mode="binary_mask"
)
print("SAM1 ViT-H automatic generator loaded. No model weights are updated.")

## 04 — Frozen SAM1 inference and resumable tile outputs

Every 40 m core is processed with 60 m RGB context on all sides. **The only spatial selection rule applied to generated polygons is centroid within that core**—not within the ROI.

This notebook saves each finished core to GeoJSON before starting the next. A new versioned directory prevents accidental reuse of cached files from the previous filtering strategy.

The two whole-area exports are `roiX_author_style.gpkg` and `roiX_nms_060.gpkg`. They deliberately include some polygons whose centroids are outside the ROI; Notebook 00 will determine which contribute to each metric.

In [ ]:
# 04 — Inference engine; saved tile outputs are the resumable checkpoint
def rgb_uint8(ds, window):
    arr=ds.read([1,2,3],window=window,boundless=True,fill_value=0)
    arr=np.moveaxis(arr,0,-1)
    if arr.dtype==np.uint8: return np.ascontiguousarray(arr)
    arr=arr.astype(np.float32)
    result=np.empty_like(arr,dtype=np.uint8)
    for k in range(3):
        band=arr[:,:,k]
        good=np.isfinite(band)
        lo,hi=np.percentile(band[good],[2,98]) if good.any() else (0,1)
        result[:,:,k]=np.uint8(255*np.clip((band-lo)/max(hi-lo,1e-9),0,1))
    return np.ascontiguousarray(result)

def mask_polygons(mask, transform):
    # Each disjoint polygon part is a potential crown; keep at most 3 largest
    # parts of each mask (historic recovered setting).
    candidates=[]
    binary=np.asarray(mask,dtype=np.uint8)
    for geometry, value in shapes(binary, mask=binary.astype(bool), transform=transform):
        if int(value)!=1: continue
        polygon=cleaned_polygon(shape(geometry))
        if polygon.geom_type=="MultiPolygon":
            candidates.extend(list(polygon.geoms))
        elif not polygon.is_empty:
            candidates.append(polygon)
    return sorted(candidates,key=lambda g:g.area,reverse=True)[:CONFIG["max_components"]]

def process_core(ds, core):
    pad=CONFIG["context_m"]
    xmin,ymin,xmax,ymax=core.bounds
    context=(xmin-pad,ymin-pad,xmax+pad,ymax+pad)
    window=from_bounds(*context,transform=ds.transform).round_offsets().round_lengths()
    image=rgb_uint8(ds,window)
    affine=ds.window_transform(window)
    masks=generator.generate(image)
    extracted=[]
    for m in masks:
        score=float(m.get("predicted_iou",np.nan))
        stability=float(m.get("stability_score",np.nan))
        for polygon in mask_polygons(m["segmentation"],affine):
            area=polygon.area
            if not (CONFIG["min_pred_area_m2"]<=area<=CONFIG["max_pred_area_m2"]):
                continue
            center=polygon.centroid
            if not core.covers(center):
                continue
            extracted.append({
                "geometry":polygon,"score":score,"stability":stability,
                "area_m2":float(area)
            })
    return extracted, len(masks)

def empty_crowns():
    return gpd.GeoDataFrame(
        {"score":pd.Series(dtype=float),"stability":pd.Series(dtype=float),
         "area_m2":pd.Series(dtype=float)},
        geometry=gpd.GeoSeries([],crs=CRS),crs=CRS)

def run_roi_tiles(roi_id):
    region=cleaned_polygon(rois.geometry.iloc[roi_id])
    cores=grid_for_roi(roi_id)
    directory=OUTPUT/f"roi{roi_id}_tiles"
    directory.mkdir(parents=True,exist_ok=True)
    tic=time.monotonic()
    with rasterio.open(RGB_PATH) as ds:
        for i,core in enumerate(cores):
            dest=directory/f"core_{i:03d}.geojson"
            if dest.exists():
                print(f"ROI {roi_id} core {i+1}/{len(cores)}: cached",flush=True)
                continue
            t=time.monotonic()
            extracted,n_masks=process_core(ds,core)
            crowns=(gpd.GeoDataFrame(extracted,geometry="geometry",crs=CRS)
                    if extracted else empty_crowns())
            tmp=dest.with_name(dest.stem+".partial.geojson")
            # GeoJSON can hold an empty FeatureCollection.
            if len(crowns):
                crowns.to_file(tmp,driver="GeoJSON")
                os.replace(tmp,dest)
            else:
                # Empty tile marker, so restart won't recompute it.
                with open(tmp,"w") as h:
                    json.dump({"type":"FeatureCollection","features":[]},h)
                os.replace(tmp,dest)
            print(f"ROI {roi_id} core {i+1}/{len(cores)}: "
                  f"{n_masks} SAM masks -> {len(crowns)} candidate polygons; "
                  f"{time.monotonic()-t:.0f}s/tile; "
                  f"total {((time.monotonic()-tic)/60):.1f} min",flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    print(f"ROI {roi_id}: all {len(cores)} core files complete",flush=True)
    return directory

def read_completed_tiles(roi_id):
    cores=grid_for_roi(roi_id)
    directory=OUTPUT/f"roi{roi_id}_tiles"
    files=[directory/f"core_{i:03d}.geojson" for i in range(len(cores))]
    missing=[p.name for p in files if not p.is_file()]
    if missing: raise FileNotFoundError(f"ROI {roi_id} missing tile files: {missing[:10]}")
    frames=[]
    for p in files:
        # Empty GeoJSONs are intentionally skipped
        with open(p) as f:
            obj=json.load(f)
        if not obj.get("features"):continue
        g=gpd.read_file(p)
        # GeoJSON format may declare WGS84 even with projected geometry;
        # verify original XY footprint before CRS override.
        if not box(*g.total_bounds).intersects(box(*tuple(BOUNDS))):
            raise ValueError(f"Suspicious tile coordinates: {p}")
        g=g.set_crs(CRS,allow_override=True)
        frames.append(g)
    if not frames:return empty_crowns()
    result=gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),crs=CRS)
    return result.reset_index(drop=True)

def nms_crowns(crowns, threshold=0.60):
    if len(crowns)==0: return crowns.copy()
    order=crowns.assign(_score=crowns.score.fillna(-1)).sort_values(
        ["_score","area_m2"],ascending=[False,False]).index.tolist()
    accepted=[]
    for j in order:
        geom=crowns.geometry.iloc[j]
        keep=True
        for k in accepted:
            prev=crowns.geometry.iloc[k]
            if not geom.intersects(prev):continue
            overlap=geom.intersection(prev).area
            union=geom.area+prev.area-overlap
            if union>0 and overlap/union>threshold:
                keep=False;break
        if keep:accepted.append(j)
    return crowns.iloc[accepted].copy().reset_index(drop=True)

def save_roi_polygons(roi_id, *, prevent_overwrite=True):
    source=read_completed_tiles(roi_id)
    final=nms_crowns(source,CONFIG["polygon_nms_iou"])
    out1=OUTPUT/f"roi{roi_id}_author_style.gpkg"
    out2=OUTPUT/f"roi{roi_id}_nms_060.gpkg"
    for dest in (out1,out2):
        if prevent_overwrite and dest.exists():
            raise FileExistsError(f"Will not overwrite existing output: {dest}")
    for g,dest in ((source,out1),(final,out2)):
        if len(g):
            g.to_file(dest,driver="GPKG")
        else:
            print("No polygons; GeoPackage not written:",dest)
    print(f"ROI {roi_id}: {len(source)} author-style -> {len(final)} NMS polygons; saved {OUTPUT}")
    return source,final

print("Tile inference helpers ready. No SAM inference has run yet.")

## 05 — Pilot ROI 1, using the revised selection rule

This requires new inference because the previous cached tile GeoJSONs had already removed crowns centred outside the ROI. A **fresh output version** has been created, so the previous pilot is preserved.

The raw exported polygon count may now exceed the historical Block 53 count due to the halo. **Compare ROI-centred subsets in Section 06**, not the complete halo-inclusive file counts.

In [ ]:
# 05 — Pilot ROI 1, save to dedicated reproduction output
pilot_tiles = run_roi_tiles(PILOT_ROI)
# Resume safely if this new-version pilot has already been exported.
pilot_files=[OUTPUT/f"roi{PILOT_ROI}_{name}.gpkg"
             for name in ("author_style","nms_060")]
if all(p.exists() for p in pilot_files):
    pilot_author=gpd.read_file(pilot_files[0]).to_crs(CRS)
    pilot_nms=gpd.read_file(pilot_files[1]).to_crs(CRS)
    print("Loaded existing pilot GeoPackages; no files overwritten.")
elif any(p.exists() for p in pilot_files):
    raise RuntimeError("Partial pilot output: inspect files before rerunning exports.")
else:
    pilot_author, pilot_nms = save_roi_polygons(PILOT_ROI)
print("Pilot complete. Compare reproduction below before accepting it.")

## 06 — Historical reproduction comparison (ROI-centred crowns only)

The new dataset includes an exterior 40 m halo, so whole-file polygon counts **must not** be compared directly with the historical ROI-filtered archive.

This audit compares polygons **whose centroids lie inside ROI 1** and plots the new footprints against the historical Block 53 NMS. This is a *reproduction diagnostic*, not a requirement to exactly replicate the historical ROI-centroid filtering. Review before approving the all-ROI production run.

In [ ]:
# 06 — Benchmark reproduced ROI 1 against frozen historical Block 53
def polygon_iou(a,b):
    if not a.intersects(b):return 0.
    inter=a.intersection(b).area
    union=a.area+b.area-inter
    return float(inter/union) if union else 0.

def similarity_report(ours,baseline):
    baseline=ensure_crs(baseline,CRS,BOUNDS)
    if not len(ours) or not len(baseline):
        return {"our_n":len(ours),"archived_n":len(baseline),
                "matched_ge_0.5":0,"matched_ge_0.9":0,
                "median_best_IoU":np.nan}
    matches=[]
    tree=baseline.sindex
    for g in ours.geometry:
        idx=tree.query(g,predicate="intersects")
        matches.append(max((polygon_iou(g,baseline.geometry.iloc[int(j)])
                            for j in idx),default=0.))
    return {"our_n":len(ours),"archived_n":len(baseline),
            "matched_ge_0.5":int(np.sum(np.array(matches)>=0.5)),
            "matched_ge_0.9":int(np.sum(np.array(matches)>=0.9)),
            "median_best_IoU":float(np.median(matches)),
            "fraction_ge_0.9":float(np.mean(np.array(matches)>=0.9))}

AUDIT=[]
for method,ours in (("author_style",pilot_author),("nms_060",pilot_nms)):
    archived_path=SOURCE/f"roi{PILOT_ROI}_{method}.gpkg"
    archive=gpd.read_file(archived_path).to_crs(CRS)
    AUDIT.append({"variant":method,"halo_total":len(full),**similarity_report(ours,archive)})
AUDIT=pd.DataFrame(AUDIT)
display(AUDIT.round(3))
print("Historical ROI 1: author-style n=142, NMS n=125.")
if (AUDIT["fraction_ge_0.9"] >= 0.90).all():
    print("Strong geometric match (still review plots).")
else:
    print("Check geometric overlap and counts carefully. No automatic batch authorization.")
print("After inspecting the pilot, explicitly set RUN_ALL_ROIS=True and PILOT_APPROVED=True in Section 01.")

# Visual side-by-side of saved baseline and reproduced pilot, with context RGB.
# It is deliberately limited to the same ROI footprint.
import matplotlib.pyplot as plt
from rasterio.windows import from_bounds
region=rois.geometry.iloc[PILOT_ROI]
with rasterio.open(RGB_PATH) as ds:
    window=from_bounds(*region.bounds,transform=ds.transform).round_offsets().round_lengths()
    window=window.intersection(Window(0,0,ds.width,ds.height))
    rgb=ds.read([1,2,3],window=window,out_shape=(3,1200,1200))
    bb=rasterio.windows.bounds(window,ds.transform)
im=np.moveaxis(rgb,0,-1).astype(float)
low,high=np.nanpercentile(im,[2,98])
im=np.clip((im-low)/(high-low+1e-9),0,1)
fig,axes=plt.subplots(1,2,figsize=(15,7),constrained_layout=True)
historical=gpd.read_file(SOURCE/f"roi{PILOT_ROI}_nms_060.gpkg").to_crs(CRS)
for ax,(label,crowns) in zip(axes,[
    ("Archived Block 53 NMS",historical),
    ("New reproduction NMS (halo-inclusive)",pilot_nms)
]):
    ax.imshow(im,extent=(bb[0],bb[2],bb[1],bb[3]))
    if len(crowns):
        crowns.boundary.plot(ax=ax,color="red",linewidth=0.8)
    ax.set_xlim(bb[0],bb[2]);ax.set_ylim(bb[1],bb[3])
    ax.set_aspect("equal");ax.set_title(f"{label}: {len(crowns)} polygons")
plt.show()


### 06B — Optional CPU-only missing-historical-crown audit

For a detailed comparison, this optional section identifies archived crown polygons not matched to the ROI-centred subset of the newly reproduced ROI 1 output. It tests proximity to the ROI boundary and the new 40 m core-grid boundaries.

**Run without rerunning SAM:** this code reads saved GeoPackages and works on CPU. It does not write output or authorize all-ROI inference.

**Note:** the new run includes a 40 m halo. Exterior polygons are not included in historical count comparisons, even though they remain available for ecological canopy-cover calculations.

In [ ]:
# 06B — Reverse audit: historical crowns missing from the reproduced ROI 1
# CPU-only; reads existing GPKG files, with no SAM inference or file writes.
from scipy.optimize import linear_sum_assignment
from shapely.geometry import box
from rasterio.windows import from_bounds, Window
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio

# Minimal bootstrap for a fresh CPU runtime; existing sessions reuse their paths.
from pathlib import Path
from shapely import make_valid
from google.colab import drive
if "SOURCE" not in globals() or "OUTPUT" not in globals() or "rois" not in globals():
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
    SOURCE = ROOT / "KVO_work/KVO_06_UNet_guided_SAM3/experiments_v53_luki_sam_only_replication"
    OUTPUT = ROOT / "V2/Crown/sam1_core_centroid_halo40_v2"
    RGB_PATH = ROOT / "data/Geo_Ref/Baego_georef.tif"
    with rasterio.open(RGB_PATH) as ds:
        CRS, BOUNDS = ds.crs, ds.bounds
    rois = gpd.read_file(ROOT / "data/Geo_Ref/Annotations/ROI_Crowns_Kvo.shp")
    if rois.crs is None:
        bx = rois.total_bounds; rb = tuple(BOUNDS)
        if not (bx[2]>=rb[0] and bx[0]<=rb[2] and bx[3]>=rb[1] and bx[1]<=rb[3]):
            raise ValueError("ROI layer has no CRS and cannot be safely aligned")
        rois = rois.set_crs(CRS)
    else:
        rois = rois.to_crs(CRS)

PILOT_ID = 1
MATCH_IOU = 0.5
EDGE_BUFFER_M = 2.0
N_ZOOMS = 12

def valid_geometry(g):
    if g is None or g.is_empty:
        return g
    return make_valid(g) if not g.is_valid else g

def safe_iou(a, b):
    if a is None or b is None or a.is_empty or b.is_empty:
        return 0.0
    a, b = valid_geometry(a), valid_geometry(b)
    if not a.intersects(b):
        return 0.0
    intersection = a.intersection(b).area
    union = a.area + b.area - intersection
    return float(intersection / union) if union > 0 else 0.0

def one_to_one_pairs(original, reproduced, min_iou=0.5):
    similarities = np.zeros((len(original), len(reproduced)), dtype=np.float32)
    if len(original) and len(reproduced):
        target_idx = reproduced.sindex
        for i, geom in enumerate(original.geometry):
            for j in target_idx.query(geom, predicate="intersects"):
                similarities[i, j] = safe_iou(geom, reproduced.geometry.iloc[int(j)])
        # Add zero-weight dummy matches to prevent a below-threshold pairing
        # from consuming a polygon that might otherwise pass the threshold.
        weighted = np.where(similarities >= min_iou, similarities, 0.)
        rows, cols = linear_sum_assignment(weighted, maximize=True)
        matches = [(int(i), int(j), float(similarities[i, j]))
                   for i, j in zip(rows, cols) if similarities[i, j] >= min_iou]
    else:
        matches = []
    matched_i = {i for i, j, score in matches}
    matched_j = {j for i, j, score in matches}
    return matches, [i for i in range(len(original)) if i not in matched_i], [
        j for j in range(len(reproduced)) if j not in matched_j], similarities

def grid_edge_distance(point, roigeom, step_m=40.):
    # Grid reconstructed from Section 02's ROI bbox origin; historical grid
    # may differ, so this is distance to NEW grid boundaries only.
    xmin, ymin, xmax, ymax = roigeom.bounds
    x, y = point.x, point.y
    dx = abs((x - xmin) % step_m)
    dy = abs((y - ymin) % step_m)
    return float(min(dx, step_m - dx, dy, step_m - dy))

roi_geometry = valid_geometry(rois.geometry.iloc[PILOT_ID])
records = []
MISSING = {}
SUMMARY = []
for variant in ("author_style", "nms_060"):
    archived_path = SOURCE / f"roi{PILOT_ID}_{variant}.gpkg"
    reproduced_path = OUTPUT / f"roi{PILOT_ID}_{variant}.gpkg"
    for p in (archived_path, reproduced_path):
        if not p.exists():
            raise FileNotFoundError(f"{p}: run the ROI 1 pilot first")
    archived = gpd.read_file(archived_path).to_crs(CRS).reset_index(drop=True)
    reproduced_all = gpd.read_file(reproduced_path).to_crs(CRS)\n    reproduced = reproduced_all.loc[reproduced_all.geometry.centroid.within(roi_geometry)].reset_index(drop=True)
    matches, misses, extras, sim = one_to_one_pairs(archived, reproduced, MATCH_IOU)
    MISSING[variant] = (archived, reproduced, misses, extras)
    for i in misses:
        geom = valid_geometry(archived.geometry.iloc[i])
        closest = int(np.argmax(sim[i])) if len(reproduced) else None
        best_iou = float(sim[i, closest]) if closest is not None else 0.
        nearest_distance = min((geom.distance(g) for g in reproduced.geometry), default=np.nan)
        centroid = geom.centroid
        roi_edge_distance = float(centroid.distance(roi_geometry.boundary))
        tile_edge_distance = grid_edge_distance(centroid, roi_geometry)
        records.append({
            "variant": variant, "archived_index": int(i),
            "area_m2": round(float(geom.area), 2),
            "best_reproduced_IoU": round(best_iou, 3),
            "closest_reproduced_index": closest,
            "nearest_geom_distance_m": round(float(nearest_distance), 2),
            "centroid_inside_ROI": bool(roi_geometry.covers(centroid)),
            "centroid_to_ROI_edge_m": round(roi_edge_distance, 2),
            "centroid_to_new_grid_edge_m": round(tile_edge_distance, 2),
            "near_ROI_edge_2m": roi_edge_distance <= EDGE_BUFFER_M,
            "near_new_grid_edge_2m": tile_edge_distance <= EDGE_BUFFER_M,
        })
    SUMMARY.append({
        "variant": variant, "archived": len(archived), "reproduced": len(reproduced),
        "one_to_one_matches": len(matches),
        "archived_unmatched": len(misses), "reproduced_unmatched": len(extras),
        "near_ROI_edge_2m": sum(x["near_ROI_edge_2m"] for x in records if x["variant"] == variant),
        "near_new_grid_edge_2m": sum(x["near_new_grid_edge_2m"] for x in records if x["variant"] == variant),
    })

MISSING_SUMMARY = pd.DataFrame(SUMMARY)
MISSING_DETAIL = pd.DataFrame(records)
print("=== Reciprocal, one-to-one polygon matching, IoU >= 0.5 ===")
display(MISSING_SUMMARY)
print("\n=== Missing archived polygons: sizes and edge proximity ===")
if MISSING_DETAIL.empty:
    print("No missing historical polygons.")
else:
    display(MISSING_DETAIL.sort_values(["variant","area_m2"],ascending=[True,False]))
    print("\nBoundary diagnostics for missing polygons:")
    display(MISSING_DETAIL.groupby("variant",as_index=False).agg(
        n_missing=("archived_index","count"),
        median_area_m2=("area_m2","median"),
        median_ROI_boundary_distance_m=("centroid_to_ROI_edge_m","median"),
        median_new_grid_boundary_distance_m=("centroid_to_new_grid_edge_m","median"),
        near_ROI_edge_2m=("near_ROI_edge_2m","sum"),
        near_new_grid_edge_2m=("near_new_grid_edge_2m","sum")))

# RGB overview: missing historical crowns marked in white on top of
# the reproduced contours (cyan), for BOTH variants.
with rasterio.open(RGB_PATH) as ds:
    w = from_bounds(*roi_geometry.bounds, transform=ds.transform)
    w = w.round_offsets().round_lengths().intersection(Window(0,0,ds.width,ds.height))
    rgb = ds.read([1,2,3], window=w, out_shape=(3,1300,1300))
    bbox = rasterio.windows.bounds(w, ds.transform)
img=np.moveaxis(rgb,0,-1).astype(float)
lo,hi=np.nanpercentile(img,[2,98])
img=np.clip((img-lo)/max(hi-lo,1e-9),0,1)
fig,axes=plt.subplots(1,2,figsize=(16,7),constrained_layout=True)
for ax,variant in zip(axes,("author_style","nms_060")):
    original, new, missing, extra = MISSING[variant]
    ax.imshow(img,extent=(bbox[0],bbox[2],bbox[1],bbox[3]))
    if len(new):
        new.boundary.plot(ax=ax,color="cyan",linewidth=.6)
    if len(missing):
        original.iloc[missing].boundary.plot(ax=ax,color="white",linewidth=2.3)
    ax.set_xlim(bbox[0],bbox[2]);ax.set_ylim(bbox[1],bbox[3])
    ax.set_title(f"{variant}: missing {len(missing)} archived polygons (white)\nReproduced polygons (cyan)")
    ax.set_aspect("equal")
plt.show()

# Show a limited number of missing historical NMS polygons in close-up.
variant="nms_060"
old, new, indices, extra = MISSING[variant]
for i in indices[:N_ZOOMS]:
    crown = old.geometry.iloc[i]
    x0,y0,x1,y1=crown.bounds
    cx,cy=(x0+x1)/2,(y0+y1)/2
    half=max((x1-x0)/2,(y1-y0)/2)+8.
    view=box(cx-half,cy-half,cx+half,cy+half)
    with rasterio.open(RGB_PATH) as ds:
        w=from_bounds(*view.bounds,transform=ds.transform)
        w=w.intersection(Window(0,0,ds.width,ds.height))
        rgb=ds.read([1,2,3],window=w,out_shape=(3,480,480))
        bb=rasterio.windows.bounds(w,ds.transform)
    tile=np.moveaxis(rgb,0,-1).astype(float)
    p2,p98=np.nanpercentile(tile,[2,98])
    tile=np.clip((tile-p2)/max(p98-p2,1e-9),0,1)
    fig,ax=plt.subplots(figsize=(5,5))
    ax.imshow(tile,extent=(bb[0],bb[2],bb[1],bb[3]))
    near=new.loc[new.geometry.intersects(view)]
    if len(near):near.boundary.plot(ax=ax,color="cyan",linewidth=1.)
    gpd.GeoSeries([crown],crs=CRS).boundary.plot(ax=ax,color="white",linewidth=2.)
    ax.set_xlim(bb[0],bb[2]);ax.set_ylim(bb[1],bb[3])
    ax.set_title(f"Archived crown {i} absent from reproduced NMS")
    ax.set_xticks([]);ax.set_yticks([])
    plt.show()

print("\nDecision: compare missing crown sizes and edge locations. Do NOT infer the cause from proximity alone.")


## 07 — Consistent six-ROI inference (explicit approval gate)

Once the revised ROI 1 pilot passes inspection, set **both** `RUN_ALL_ROIS=True` and `PILOT_APPROVED=True` in Section 01, rerun Section 01 and Section 07.

This processes **all six ROIs (0–5)** with identical settings. ROI 1 already completed in Section 05 will be reused; ROI 4 will be freshly inferred, rather than mixing historical ROI 4 polygons with new predictions.

Each core prints progress; rerunning safely skips completed tiles and existing exports. The previous `sam1_block53_reproduction/` results remain untouched.

In [ ]:
# 07 — All six ROIs; gated, resumable, and consistent
if not RUN_ALL_ROIS or not PILOT_APPROVED:
    print("No all-ROI run started. Set RUN_ALL_ROIS=True and PILOT_APPROVED=True in Section 01 after reviewing the pilot.")
else:
    for roi_id in ALL_ROIS:
        print(f"\n===== ROI {roi_id} | core-centroid + 40m halo =====",flush=True)
        run_roi_tiles(roi_id)
        out1=OUTPUT/f"roi{roi_id}_author_style.gpkg"
        out2=OUTPUT/f"roi{roi_id}_nms_060.gpkg"
        if out1.exists() and out2.exists():
            print(f"ROI {roi_id}: both GeoPackages already saved; preserving files.")
        elif out1.exists() or out2.exists():
            raise RuntimeError(f"ROI {roi_id}: only one GeoPackage exists; inspect before rerun.")
        else:
            save_roi_polygons(roi_id)
    print("\nSix-ROI processing complete. Review files before ecological evaluation.")

## 08 — Feed all six consistent predictions into Notebook 00

When the run is complete, update Notebook 00 Section 01 to use the new `V2/Crown/sam1_core_centroid_halo40_v2/` outputs for **all six ROIs**, rather than combining Block 53 ROI 1/4 with the new outputs.

```python
OUT = ROOT / "V2/Crown/sam1_core_centroid_halo40_v2"
EXTRA_SAM1_FILES = {
    (roi, method): OUT / f"roi{roi}_{method}.gpkg"
    for roi in range(6)
    for method in ("author_style", "nms_060")
}
```

Notebook 00 will then count crowns by **centroid within each ROI**, but clip *every intersecting saved polygon* for summed CPA and union canopy cover. This is why we retained the exterior halo polygons.

The original Block 53 reference experiment and the first `sam1_block53_reproduction` pilot remain separate, unchanged and fully accessible.

**Scientific caution:** This is a consistently applied new inference protocol derived from the historical settings, not a perfect archival reproduction. Interpret differences with human annotations only after assessing their spatial completeness.